# V9 GDE-Net Final Ablation Summary

Reads available GDE-Net ablation result CSV files from Kaggle working output and builds the final comparison tables.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd

BASE_REPORT_ROOT = "/kaggle/working/report_EfficientNetB4"
OUT_DIR = os.path.join(BASE_REPORT_ROOT, "final_gdenet_ablation_summary")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Reading reports from: {BASE_REPORT_ROOT}")


In [ ]:
strategy_files = sorted(glob.glob(os.path.join(BASE_REPORT_ROOT, "*", "strategy_summary.csv")))
if not strategy_files:
    raise FileNotFoundError("No strategy_summary.csv files found. Run the ablation notebooks first.")

run_frames = []
for fpath in strategy_files:
    df = pd.read_csv(fpath)
    df["source_file"] = fpath
    run_frames.append(df)

runs_df = pd.concat(run_frames, ignore_index=True)
print(f"Loaded {len(runs_df)} run rows from {len(strategy_files)} strategy summaries.")
runs_df.head()


In [ ]:
metric_cols = ["accuracy", "precision", "recall", "f1_score", "auc_macro", "auc_weighted"]
summary_rows = []
for (strategy_key, strategy_label), g in runs_df.groupby(["strategy_key", "strategy_label"], dropna=False):
    row = {
        "Model": strategy_label,
        "strategy_key": strategy_key,
        "n_runs": len(g),
    }
    for col in metric_cols:
        vals = g[col].astype(float).values
        row[col] = float(np.nanmean(vals))
        row[f"{col}_std"] = float(np.nanstd(vals))
        row[f"{col}_mean_std"] = f"{np.nanmean(vals):.4f} +/- {np.nanstd(vals):.4f}"
    summary_rows.append(row)

overall_summary = pd.DataFrame(summary_rows).sort_values("f1_score", ascending=False)
overall_summary.to_csv(os.path.join(OUT_DIR, "final_ablation_summary.csv"), index=False)
print("Overall summary:")
print(overall_summary[["Model", "accuracy_mean_std", "precision_mean_std", "recall_mean_std", "f1_score_mean_std", "auc_macro_mean_std", "auc_weighted_mean_std"]].to_string(index=False))


In [ ]:
baseline_candidates = overall_summary[overall_summary["strategy_key"].str.contains("v0|cbloss_baseline|cbloss_only", case=False, na=False)]
if len(baseline_candidates) == 0:
    baseline = overall_summary.iloc[-1]
    print("Baseline key not found; using the last row as fallback baseline.")
else:
    baseline = baseline_candidates.iloc[0]

baseline_metrics = {col: baseline[col] for col in metric_cols}
delta_df = overall_summary[["Model", "strategy_key"]].copy()
for col in metric_cols:
    delta_df[f"delta_{col}"] = overall_summary[col] - baseline_metrics[col]

delta_df.to_csv(os.path.join(OUT_DIR, "final_ablation_delta_vs_baseline.csv"), index=False)
print(f"Baseline: {baseline['Model']}")
print(delta_df.to_string(index=False))


In [ ]:
def component_flags(strategy_key):
    key = str(strategy_key).lower()
    return {
        "GAP+GMP": "v1" in key or "gap_gmp" in key or "gdenet" in key or "evidence" in key,
        "Coverage": "coverage" in key or "dual" in key or "gdenet" in key,
        "Defect": "defect" in key or "dual" in key or "gdenet" in key,
        "Gate": "coverage" in key or "defect" in key or "dual" in key or "gdenet" in key,
        "Aux Loss": False,
        "Diversity": "diversity" in key,
        "Loss": "CE" if key.endswith("_ce") or "categoricalcrossentropy" in key else "CBLoss",
    }

component_rows = []
for _, row in overall_summary.iterrows():
    flags = component_flags(row["strategy_key"])
    flags.update({"Model": row["Model"], "F1": row["f1_score"]})
    component_rows.append(flags)

component_df = pd.DataFrame(component_rows)
component_df.to_csv(os.path.join(OUT_DIR, "component_ablation_table.csv"), index=False)
print(component_df.to_string(index=False))


In [ ]:
per_class_files = sorted(glob.glob(os.path.join(BASE_REPORT_ROOT, "*", "per_class_metrics_summary.csv")))
per_class_frames = []
for fpath in per_class_files:
    strategy_key = os.path.basename(os.path.dirname(fpath))
    df = pd.read_csv(fpath)
    df["strategy_key"] = strategy_key
    per_class_frames.append(df)

if per_class_frames:
    per_class_df = pd.concat(per_class_frames, ignore_index=True)
    per_class_df.to_csv(os.path.join(OUT_DIR, "final_per_class_ablation_summary.csv"), index=False)
    print(f"Saved per-class summary rows: {len(per_class_df)}")
else:
    print("No per_class_metrics_summary.csv files found yet.")


In [ ]:
xlsx_path = os.path.join(OUT_DIR, "final_ablation_summary.xlsx")
try:
    with pd.ExcelWriter(xlsx_path) as writer:
        overall_summary.to_excel(writer, sheet_name="overall", index=False)
        delta_df.to_excel(writer, sheet_name="delta_vs_baseline", index=False)
        component_df.to_excel(writer, sheet_name="components", index=False)
        if "per_class_df" in globals():
            per_class_df.to_excel(writer, sheet_name="per_class", index=False)
    print(f"Saved workbook -> {xlsx_path}")
except Exception as exc:
    print(f"Excel export skipped: {exc}")
